# 🤖 Experimentos de Machine Learning Clássico (v3 – Wavelet Stats + Raw Coefs)

## Objetivo
Avaliar modelos de ML clássico com **features wavelet** (db2):
- **LinearSVR** (Linear Support Vector Regression)
- **SGDRegressor** (Stochastic Gradient Descent – epsilon insensitive)
- **ElasticNet** (L1 + L2 Regularization)
- **Random Forest**
- **XGBoost**
- **LightGBM**
- **CatBoost**
- **Stacking Ensemble** (RF + XGB + LGB + CatBoost → Ridge)

## Melhorias sobre v1
1. **Features**: wavelet stats (12 × 3 sub-bandas) + coeficientes brutos
2. **Otimização via RandomizedSearchCV** (mais rápida que GridSearch)
3. **Feature selection**: VarianceThreshold + Mutual Information
4. **Paralelização agressiva** (n_jobs = CPU - 2 para estabilidade)
5. **Modelos lineares como baseline** + ensemble stacking para teto de desempenho

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import os
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.svm import LinearSVR
from sklearn.linear_model import SGDRegressor, ElasticNet, RidgeCV
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression, VarianceThreshold
from sklearn.base import clone
from scipy.stats import randint, uniform, loguniform
from scipy import stats as sp_stats
from joblib import Parallel, delayed
import pywt
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Imports locais
import sys
sys.path.append('.')
from src.evaluation import RegressionEvaluator, ResultsManager
from src.visualization import ExperimentVisualizer
from config.experiment_config import (
    DATA_DIR, RESULTS_DIR, WAVELET_CONFIG, SEED,
    N_JOBS, ML_MODELS_CONFIG, ML_SEARCH_CONFIG, ML_FEATURE_CONFIG,
    ML_VIS_CONFIG, build_param_dist,
)

# Configuração
plt.style.use('seaborn-v0_8-whitegrid')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Imports realizados com sucesso!")
print(f"   CPUs disponíveis: {os.cpu_count()}, usando N_JOBS={N_JOBS}")

## 1. Carregar Dados

In [ ]:
# Carregar datasets
X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

print(f"📦 Dados Carregados:")
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Val:   X={X_val.shape}, y={y_val.shape}")
print(f"  Test:  X={X_test.shape}, y={y_test.shape}")

## 2. Extração de Features (Wavelet Stats + Coeficientes Brutos)

In [ ]:
# ============================================================
# Feature Extraction: Wavelet Stats + Raw Coefficients
# ============================================================
WAVELET = WAVELET_CONFIG['wavelet_type']
LEVEL  = WAVELET_CONFIG['decomposition_level']
MODE   = WAVELET_CONFIG['mode']

def extract_enhanced_features_single(x):
    """
    Features de um único sinal:
      1) 12 stats × (level+1) sub-bandas wavelet
      2) Coeficientes wavelet brutos concatenados
    """
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 1:
        x = x[:, None]
    feats = []
    for ch in range(x.shape[1]):
        xc = x[:, ch]

        # ---- 1) Wavelet stats por sub-banda (canal ch) ----
        coeffs = pywt.wavedec(xc, WAVELET, mode=MODE, level=LEVEL)
        for c in coeffs:
            feats.extend([
                np.mean(c), np.std(c), np.var(c),
                np.max(c), np.min(c), np.ptp(c),
                np.median(c),
                np.sum(c ** 2),                               # energy
                float(sp_stats.skew(c)),
                float(sp_stats.kurtosis(c)),
                np.sqrt(np.mean(c ** 2)),                     # RMS
                np.percentile(c, 75) - np.percentile(c, 25),  # IQR
            ])

        # ---- 2) Coeficientes wavelet brutos (canal ch) ----
        feats.extend(np.concatenate(coeffs).tolist())

    return np.array(feats, dtype=np.float32)


def extract_enhanced_features(X, n_jobs=N_JOBS):
    """Extrai features em paralelo (joblib threads)."""
    results = Parallel(n_jobs=n_jobs, prefer='threads', verbose=0)(
        delayed(extract_enhanced_features_single)(X[i]) for i in range(len(X))
    )
    return np.vstack(results)


def get_feature_names(sample_len=256):
    """Gera nomes das features (mesma ordem de extract_enhanced_features_single)."""
    names = []
    sub_bands = [f'A{LEVEL}'] + [f'D{LEVEL - i}' for i in range(LEVEL)]
    stat_names = ['mean', 'std', 'var', 'max', 'min', 'ptp', 'median',
                  'energy', 'skew', 'kurtosis', 'rms', 'iqr']
    for sb in sub_bands:
        for s in stat_names:
            names.append(f'{sb}_{s}')
    dummy = pywt.wavedec(np.zeros(sample_len), WAVELET, mode=MODE, level=LEVEL)
    n_raw = sum(len(c) for c in dummy)
    for i in range(n_raw):
        names.append(f'raw_coeff_{i}')
    return names


# ---- Executar extração ----
print(f"🔧 Config Wavelet: {WAVELET}, level={LEVEL}, mode={MODE}")
print(f"   Paralelização: {N_JOBS} threads\n")

t0 = time.time()
print("Extraindo features TRAIN...")
X_train_feat = extract_enhanced_features(X_train)
print("Extraindo features VAL...")
X_val_feat = extract_enhanced_features(X_val)
print("Extraindo features TEST...")
X_test_feat = extract_enhanced_features(X_test)

elapsed_feat = time.time() - t0
n_sub = LEVEL + 1
n_stat = n_sub * 12
n_raw = X_train_feat.shape[1] - n_stat
print(f"\n⏱️  Tempo de extração: {elapsed_feat:.1f}s")
print(f"\n📊 Features: {X_train_feat.shape[1]} por amostra")
print(f"    Wavelet stats:       {n_stat}")
print(f"    Raw coefficients:    {n_raw}")
print(f"\n  Train: {X_train_feat.shape}")
print(f"  Val:   {X_val_feat.shape}")
print(f"  Test:  {X_test_feat.shape}")

In [ ]:
# ============================================================
# Normalização + Feature Selection leve
# ============================================================

# Remover features com variância nula
vt = VarianceThreshold(threshold=ML_FEATURE_CONFIG['variance_threshold'])
X_train_vt = vt.fit_transform(X_train_feat)
X_val_vt = vt.transform(X_val_feat)
X_test_vt = vt.transform(X_test_feat)
n_removed = X_train_feat.shape[1] - X_train_vt.shape[1]
print(f"VarianceThreshold: {X_train_vt.shape[1]} features mantidas ({n_removed} removidas)")

# Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_vt)
X_val_scaled = scaler.transform(X_val_vt)
X_test_scaled = scaler.transform(X_test_vt)

# Safety: remover NaN/Inf
X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_val_scaled = np.nan_to_num(X_val_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# Combinar train + val para busca de hiperparâmetros
X_trainval = np.vstack([X_train_scaled, X_val_scaled])
y_trainval = np.concatenate([y_train, y_val])

# Mutual Information para análise
mi_subsample = ML_FEATURE_CONFIG['mi_subsample']
mi_top_k = ML_FEATURE_CONFIG['mi_top_k']
mi_n_neighbors = ML_FEATURE_CONFIG['mi_n_neighbors']
print(f"\n🔍 Calculando Mutual Information (top {mi_top_k}, sub-amostra={mi_subsample})...")
mi_scores = mutual_info_regression(
    X_train_scaled[:mi_subsample], y_train[:mi_subsample],
    n_neighbors=mi_n_neighbors, random_state=SEED
)
feature_names_all = get_feature_names()
kept_mask = vt.get_support()
feature_names_filtered = [feature_names_all[i] for i, k in enumerate(kept_mask) if k]
top_mi_idx = np.argsort(mi_scores)[::-1][:mi_top_k]
print(f"  Top {mi_top_k} features por MI:")
for rank, idx in enumerate(top_mi_idx, 1):
    name = feature_names_filtered[idx] if idx < len(feature_names_filtered) else f'feat_{idx}'
    print(f"    {rank:2d}. {name:30s} MI={mi_scores[idx]:.4f}")

print(f"\n✅ Features prontas")
print(f"  TrainVal combinado: {X_trainval.shape}")

## 3. Configuração de Experimentos

In [ ]:
# Gerenciador de resultados
results_manager = ResultsManager(RESULTS_DIR / "ml_experiments")
evaluator = RegressionEvaluator()

# Armazenar todos os resultados
all_results = {}
all_cv_results = {}   # Armazenar cv_results_ de cada RandomizedSearchCV

# TimeSeriesSplit para validação cruzada (do config)
tscv = TimeSeriesSplit(n_splits=ML_SEARCH_CONFIG['cv_splits'])

# ---- Helper para rodar e registrar um experimento ----
def run_experiment(name, model, X_tv, y_tv, X_te, y_te,
                   param_dist=None, n_iter=None,
                   category='ML_FixedWavelet'):
    """Roda RandomizedSearchCV (ou fit direto) e retorna o melhor modelo.
    
    Armazena TODOS os resultados do CV (não só o melhor) em all_cv_results.
    """
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    t0 = time.time()

    if param_dist:
        search = RandomizedSearchCV(
            model, param_dist, n_iter=n_iter, cv=tscv,
            scoring=ML_SEARCH_CONFIG['scoring'],
            n_jobs=ML_SEARCH_CONFIG['n_jobs'],
            verbose=ML_SEARCH_CONFIG['verbose'],
            random_state=ML_SEARCH_CONFIG['random_state'],
        )
        search.fit(X_tv, y_tv)
        best = search.best_estimator_
        best_params = search.best_params_
        
        # Salvar TODOS os resultados do grid/random search
        cv_df = pd.DataFrame(search.cv_results_)
        cv_df['model_name'] = name
        all_cv_results[name] = cv_df
        
        print(f"  Melhores parâmetros: {best_params}")
        print(f"  Total de combinações avaliadas: {len(cv_df)}")
    else:
        model.fit(X_tv, y_tv)
        best = model
        best_params = model.get_params()

    y_pred = best.predict(X_te)
    metrics = evaluator.evaluate(y_te, y_pred)
    elapsed = time.time() - t0

    print(f"\n  📊 {name}:")
    print(f"     RMSE: {metrics['rmse']:.6f}")
    print(f"     MAE:  {metrics['mae']:.6f}")
    print(f"     R²:   {metrics['r2']:.6f}")
    print(f"     Tempo: {elapsed:.1f}s")

    all_results[name] = {
        'metrics': metrics,
        'best_params': best_params,
        'time': elapsed,
        'y_pred': y_pred,
        'model': best,
    }
    results_manager.log_experiment(category, name, metrics, {'params': str(best_params)})
    return best

print(f"✅ Configuração pronta – RandomizedSearchCV com TimeSeriesSplit({ML_SEARCH_CONFIG['cv_splits']})")
print(f"   Paralelização: n_jobs={ML_SEARCH_CONFIG['n_jobs']}")

## 4. Experimento 1: LinearSVR

In [ ]:
cfg = ML_MODELS_CONFIG['LinearSVR']
best_linearsvr = run_experiment(
    'Wavelet_LinearSVR',
    LinearSVR(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 5. Experimento 2: SGDRegressor (epsilon-insensitive)

In [ ]:
cfg = ML_MODELS_CONFIG['SGDRegressor']
best_sgd = run_experiment(
    'Wavelet_SGD',
    SGDRegressor(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 6. Experimento 3: ElasticNet

In [ ]:
cfg = ML_MODELS_CONFIG['ElasticNet']
best_elasticnet = run_experiment(
    'Wavelet_ElasticNet',
    ElasticNet(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 7. Experimento 4: Random Forest

In [ ]:
cfg = ML_MODELS_CONFIG['RandomForest']
best_rf = run_experiment(
    'Wavelet_RF',
    RandomForestRegressor(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 8. Experimento 5: XGBoost

In [ ]:
cfg = ML_MODELS_CONFIG['XGBoost']
best_xgb = run_experiment(
    'Wavelet_XGB',
    xgb.XGBRegressor(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 9. Experimento 6: LightGBM

In [ ]:
cfg = ML_MODELS_CONFIG['LightGBM']
best_lgb = run_experiment(
    'Wavelet_LGBM',
    lgb.LGBMRegressor(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 10. Experimento 7: CatBoost

In [ ]:
cfg = ML_MODELS_CONFIG['CatBoost']
best_catboost = run_experiment(
    'Wavelet_CatBoost',
    CatBoostRegressor(**cfg['model_kwargs']),
    X_trainval, y_trainval, X_test_scaled, y_test,
    param_dist=build_param_dist(cfg['param_dist']),
    n_iter=cfg['n_iter'],
)

## 11. Experimento 8: Stacking Ensemble

In [ ]:
# Stacking ensemble: RF + XGB + LGB + CatBoost → Ridge meta-learner
print("\n" + "=" * 60)
print("  🔗 Wavelet_Stacking (RF + XGB + LGB + CatBoost → Ridge)")
print("=" * 60)

t0 = time.time()

stack_cfg = ML_MODELS_CONFIG['Stacking']
base_n_jobs = stack_cfg.get('base_n_jobs', 4)

# Reconstruir base estimators com n_jobs do config
def _set_parallel(est, n):
    est = clone(est)
    if hasattr(est, 'n_jobs'):
        est.set_params(n_jobs=n)
    if hasattr(est, 'thread_count'):
        est.set_params(thread_count=n)
    return est

# Usar KFold em vez de TimeSeriesSplit para o Stacking
# (cross_val_predict exige partições completas)
from sklearn.model_selection import KFold
stack_cv = KFold(n_splits=ML_SEARCH_CONFIG['cv_splits'], shuffle=False)

stacking = StackingRegressor(
    estimators=[
        ('rf',       _set_parallel(all_results['Wavelet_RF']['model'], base_n_jobs)),
        ('xgb',      _set_parallel(all_results['Wavelet_XGB']['model'], base_n_jobs)),
        ('lgb',      _set_parallel(all_results['Wavelet_LGBM']['model'], base_n_jobs)),
        ('catboost', _set_parallel(all_results['Wavelet_CatBoost']['model'], base_n_jobs)),
    ],
    final_estimator=RidgeCV(alphas=stack_cfg['ridge_alphas']),
    cv=stack_cv,
    n_jobs=stack_cfg['model_kwargs'].get('n_jobs', 1),
    passthrough=stack_cfg['model_kwargs'].get('passthrough', False),
)
stacking.fit(X_trainval, y_trainval)

y_pred_stack = stacking.predict(X_test_scaled)
stack_metrics = evaluator.evaluate(y_test, y_pred_stack)
elapsed = time.time() - t0

print(f"\n  📊 Wavelet_Stacking:")
print(f"     RMSE: {stack_metrics['rmse']:.6f}")
print(f"     MAE:  {stack_metrics['mae']:.6f}")
print(f"     R²:   {stack_metrics['r2']:.6f}")
print(f"     Tempo: {elapsed:.1f}s")

all_results['Wavelet_Stacking'] = {
    'metrics': stack_metrics,
    'best_params': {'meta': 'RidgeCV', 'base': stack_cfg['base_learners']},
    'time': elapsed,
    'y_pred': y_pred_stack,
    'model': stacking,
}
results_manager.log_experiment('ML_FixedWavelet', 'Stacking', stack_metrics,
                               {'base': '+'.join(stack_cfg['base_learners']), 'meta': stack_cfg['meta_learner']})

## 12. Comparação dos Resultados

In [ ]:
# Criar DataFrame comparativo (melhor de cada modelo)
comparison_data = []
for model_name, result in all_results.items():
    row = {
        'Model': model_name,
        'RMSE': result['metrics']['rmse'],
        'MAE': result['metrics']['mae'],
        'R²': result['metrics']['r2'],
        'Time (s)': result['time']
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('RMSE')

print("\n" + "=" * 80)
print("📊 COMPARAÇÃO FINAL - ML com Features Wavelet Enriquecidas (db2)")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Salvar comparação (melhor por modelo)
out_dir = RESULTS_DIR / "ml_experiments"
comparison_df.to_csv(out_dir / "comparison_ml.csv", index=False)
print(f"\n✅ CSV salvo em: {out_dir / 'comparison_ml.csv'}")

# Salvar TODOS os resultados do grid/random search
if all_cv_results:
    full_cv_df = pd.concat(all_cv_results.values(), ignore_index=True)
    full_cv_df.to_csv(out_dir / "all_search_results_ml.csv", index=False)
    print(f"✅ Todos os resultados do search salvos: {out_dir / 'all_search_results_ml.csv'}")
    print(f"   Total de combinações avaliadas: {len(full_cv_df)}")

In [ ]:
# Visualização comparativa
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

model_colors = {
    'Wavelet_LinearSVR': '#4e79a7', 'Wavelet_SGD': '#59a14f',
    'Wavelet_ElasticNet': '#76b7b2', 'Wavelet_RF': '#f28e2b',
    'Wavelet_XGB': '#e15759', 'Wavelet_LGBM': '#edc948',
    'Wavelet_CatBoost': '#b07aa1', 'Wavelet_Stacking': '#ff9da7',
}

metrics_to_plot = ['RMSE', 'MAE', 'R²']
for idx, metric in enumerate(metrics_to_plot):
    data = comparison_df.set_index('Model')[metric].sort_values(
        ascending=(metric != 'R²')
    )
    colors = [model_colors.get(m, '#999999') for m in data.index]
    bars = axes[idx].barh(data.index, data.values, color=colors)
    axes[idx].set_xlabel(metric)
    axes[idx].set_title(f'Comparação: {metric}')
    axes[idx].grid(True, alpha=0.3, axis='x')
    for bar, val in zip(bars, data.values):
        axes[idx].text(val, bar.get_y() + bar.get_height()/2,
                      f'{val:.4f}', va='center', ha='left', fontsize=8)

plt.suptitle('ML Clássico com Features Wavelet (Stats + Raw Coefs, db2)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ml_experiments" / "comparison_ml.png", dpi=150, bbox_inches='tight')
plt.show()

## 13. Análise de Predições do Melhor Modelo

In [ ]:
# Encontrar melhor modelo
best_model_name = comparison_df.iloc[0]['Model']
best_result = all_results[best_model_name]

print(f"\n🏆 Melhor Modelo: {best_model_name}")
print(f"   RMSE={best_result['metrics']['rmse']:.6f}, R²={best_result['metrics']['r2']:.6f}")

# Plot de predições
visualizer = ExperimentVisualizer()
fig = visualizer.plot_prediction_comparison(
    y_test, best_result['y_pred'],
    model_name=best_model_name,
    n_samples=ML_VIS_CONFIG['prediction_n_samples'],
    save_path=RESULTS_DIR / "ml_experiments" / f"predictions_{best_model_name}.png"
)
plt.show()

## 14. Feature Importance (modelos baseados em árvore)

In [ ]:
# Feature importance – 4 modelos baseados em árvore
top_k = ML_VIS_CONFIG['feature_importance_top_k']
fig, axes = plt.subplots(2, 2, figsize=(22, 16))

tree_models = [
    ('Random Forest', all_results['Wavelet_RF']['model']),
    ('XGBoost', all_results['Wavelet_XGB']['model']),
    ('LightGBM', all_results['Wavelet_LGBM']['model']),
    ('CatBoost', all_results['Wavelet_CatBoost']['model']),
]

for ax, (title, model) in zip(axes.flat, tree_models):
    imp = model.feature_importances_
    top_idx = np.argsort(imp)[::-1][:top_k]
    names = [feature_names_filtered[i] if i < len(feature_names_filtered)
             else f'feat_{i}' for i in top_idx]
    ax.barh(range(len(top_idx)), imp[top_idx], color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels(names, fontsize=7)
    ax.set_title(f'{title} – Top {top_k} Features')
    ax.invert_yaxis()

plt.suptitle('Importância das Features (Wavelet Stats + Raw Coefs)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ml_experiments" / "feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## 15. Resumo

In [ ]:
print("\n" + "=" * 80)
print("📋 RESUMO – Experimentos ML com Features Wavelet (Stats + Raw Coefs)")
print("=" * 80)
print(f"\n✅ Modelos avaliados: {len(all_results)}")
print(f"✅ Melhor modelo: {best_model_name}")
print(f"✅ Melhor RMSE: {comparison_df.iloc[0]['RMSE']:.6f}")
print(f"✅ Melhor R²:   {comparison_df.iloc[0]['R²']:.6f}")
print(f"\n📊 Features: {X_trainval.shape[1]} "
      f"(wavelet stats + raw coefs)")
print(f"📁 Resultados salvos em: {RESULTS_DIR / 'ml_experiments'}")
print("\n🎉 Notebook concluído com sucesso!")